# Solutions — TPC-DS Store Sales (Medium 08)

**Datasets:** `samples.tpcds_sf1.store_sales`, `date_dim`, `item`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

store_sales = spark.table("samples.tpcds_sf1.store_sales")
date_dim    = spark.table("samples.tpcds_sf1.date_dim")
item        = spark.table("samples.tpcds_sf1.item")

## Solution 1 — Total Net Paid & Transactions per Year

In [ ]:
result_1 = (
    store_sales
    .join(date_dim, store_sales["ss_sold_date_sk"] == date_dim["d_date_sk"])
    .groupBy("d_year")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_net_paid"),
        F.count("*").alias("total_transactions")
    )
    .orderBy("d_year")
)

## Solution 2 — Top 10 Items by Total Net Paid

In [ ]:
result_2 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .groupBy("i_product_name","i_category")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_net_paid"),
        F.round(F.sum("ss_quantity"), 0).alias("total_quantity")
    )
    .orderBy(F.col("total_net_paid").desc())
    .limit(10)
)

## Solution 3 — Sales, Avg Discount, Avg Net Profit per Store

In [ ]:
result_3 = (
    store_sales
    .groupBy("ss_store_sk")
    .agg(
        F.round(F.sum("ss_sales_price"), 2).alias("total_sales"),
        F.round(F.avg("ss_coupon_amt"), 2).alias("avg_discount"),
        F.round(F.avg("ss_net_profit"), 2).alias("avg_net_profit")
    )
    .orderBy(F.col("total_sales").desc())
)

## Solution 4 — Transactions with Coupons

In [ ]:
result_4 = (
    store_sales
    .filter(F.col("ss_coupon_amt") > 0)
    .select("ss_ticket_number","ss_item_sk","ss_sales_price","ss_coupon_amt","ss_net_paid")
    .orderBy(F.col("ss_coupon_amt").desc())
)

## Solution 5 — Revenue per Category per Year

In [ ]:
result_5 = (
    store_sales
    .join(date_dim, store_sales["ss_sold_date_sk"] == date_dim["d_date_sk"])
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .groupBy("d_year","i_category")
    .agg(
        F.round(F.sum("ss_net_paid"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count")
    )
    .orderBy("d_year","i_category")
)

## Solution 6 — Top 5 Items by Net Paid within Category

In [ ]:
w = Window.partitionBy("i_category").orderBy(F.col("total_net_paid").desc())
result_6 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .groupBy("i_category","i_product_name")
    .agg(F.round(F.sum("ss_net_paid"), 2).alias("total_net_paid"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 5)
    .orderBy("i_category","rank")
)

## Solution 7 — Loss-Making Items per Category

In [ ]:
result_7 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .filter(F.col("ss_net_profit") < 0)
    .groupBy("i_category")
    .agg(
        F.count("*").alias("loss_count"),
        F.round(F.sum("ss_net_profit"), 2).alias("total_loss")
    )
    .orderBy("total_loss")
)